# Project interactive visualization
Using a dataset that your group is consider using for the term project, let's build some meaningful user-driven data visualization. Depending on your dataset this could include:
- Usage of geospatial information
- Building interactive views with widgets
- Organize multiple components into a simple dashboard

Keep these design principles in mind:
While you navigate through this notebook some things to take into consideration:
* Do not add interaction just to add it. Make sure it helps answer a question.
* Use meaningful titles and labels
* Document your code so it's readable and clean. If something does not work, document the issue and explain your best attempt.

In [2]:
## These are most likely the libraries you will use
# Add or remove imports as needed
import warnings
warnings.filterwarnings("ignore")

!pip install hvplot
import hvplot.pandas as hvplot

import pandas as pd
import numpy as np

# Visualization libraries
import plotly.express as px
import plotly.graph_objects as go

# Geospatial / geocoding
from geopy.geocoders import Nominatim

# Panel
import panel as pn
pn.extension('plotly')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.6/180.6 kB 10.1 MB/s eta 0:00:00


In [3]:
### Load your dataset here

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Tech
apple_url = 'https://raw.githubusercontent.com/CS133-DataVisualization/term-project-stocksuccess/main/datasets/apple.csv'
nvidia_url = 'https://raw.githubusercontent.com/CS133-DataVisualization/term-project-stocksuccess/main/datasets/nvidia.csv'
microsoft_url = 'https://raw.githubusercontent.com/CS133-DataVisualization/term-project-stocksuccess/main/datasets/microsoft.csv'

# Defense
lockheed_url = 'https://raw.githubusercontent.com/CS133-DataVisualization/term-project-stocksuccess/main/datasets/lockheedmartin.csv'
northrop_url = 'https://raw.githubusercontent.com/CS133-DataVisualization/term-project-stocksuccess/main/datasets/northropgrumman.csv'
boeing_url = 'https://raw.githubusercontent.com/CS133-DataVisualization/term-project-stocksuccess/main/datasets/boeing.csv.zip'

# Waste Management
waste_mgt_url = 'https://raw.githubusercontent.com/CS133-DataVisualization/term-project-stocksuccess/main/datasets/wastemanagementinc.csv'
republic_url = 'https://raw.githubusercontent.com/CS133-DataVisualization/term-project-stocksuccess/main/datasets/republicservices.csv'
waste_conn_url = 'https://raw.githubusercontent.com/CS133-DataVisualization/term-project-stocksuccess/main/datasets/wasteconnections.csv'


In [4]:
# Write your code here
url_list = {
    'AAPL': apple_url,
    'NVDA': nvidia_url,
    'MSFT': microsoft_url,
    'LMT': lockheed_url,
    'NOC': northrop_url,
    'BA': boeing_url,
    'WM': waste_mgt_url,
    'RSG': republic_url,
    'WCN': waste_conn_url
}

for name, url in url_list.items():
    df = pd.read_csv(url)

    print(f"\n{name} Columns:")
    print(df.columns.tolist()) #see list of column names to find common columns

all_frames = []

rename_map = {
    'date': 'Date',
    'price': 'Close',
    'Price': 'Close',
    'Close/Last': 'Close',
    'close': 'Close',
    'open': 'Open',
    'high': 'High',
    'low': 'Low',
    'volume': 'Volume',
    'Vol.': 'Volume',
    'adj_close': 'Adj Close'
}

tech_tickers = ['AAPL', 'NVDA', 'MSFT']
defense_tickers = ['LMT', 'NOC', 'BA']
waste_tickers = ['WM', 'RSG', 'WCN']

#read csv files and store data into one dataframe
for name, url in url_list.items():
    df = pd.read_csv(url,
                     na_values=["-", ""])

    df.columns = [col.strip() for col in df.columns]

    df = df.rename(columns=rename_map)

    df['Date'] = pd.to_datetime(df['Date'], utc=True).dt.date

    df['Ticker'] = name

    #create column 'Industry' that specifies the industry
    if name in tech_tickers:
        df['Industry'] = 'Tech'
    elif name in defense_tickers:
        df['Industry'] = 'Defense'
    elif name in waste_tickers:
        df['Industry'] = 'Waste Management'

    #keep only the columns that exist in almost all sets
    standard_cols = ['Industry', 'Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume']
    df = df[standard_cols]

    all_frames.append(df)

#merge the dataframes into one
stocks = pd.concat(all_frames, ignore_index=True)



AAPL Columns:
['Date', 'Price', 'Open', 'High', 'Low', 'Vol.', 'Change %']

NVDA Columns:
['date', 'open', 'high', 'low', 'close', 'adj_close', 'volume']

MSFT Columns:
['date', 'open', 'high', 'low', 'close', 'adj_close', 'volume']

LMT Columns:
['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']

NOC Columns:
['Date', 'Close/Last', 'Volume', 'Open', 'High', 'Low']

BA Columns:
['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']

WM Columns:
['Date', 'Close/Last', 'Volume', 'Open', 'High', 'Low']

RSG Columns:
['Date', 'Close/Last', 'Volume', 'Open', 'High', 'Low']

WCN Columns:
['Date', 'Close/Last', 'Volume', 'Open', 'High', 'Low']


In [5]:
#clean 'Volume', 'Open', 'High', 'Low', and 'Close' columns
stocks['Volume'] = pd.to_numeric(stocks['Volume'].astype(str).str.replace('M', 'e6').str.replace('K', 'e3').str.replace('B', 'e9'), errors='coerce')

numeric_cols = ['Open', 'High', 'Low', 'Close']

for col in numeric_cols:
    stocks[col] = pd.to_numeric(
        stocks[col].astype(str).str.replace('$', '').str.replace(',', ''), errors='coerce')

In [6]:
stocks.head()

,Industry,Date,Ticker,Open,High,Low,Close,Volume
0,Tech,2025-09-26,AAPL,254.10,257.60,253.78,255.46,46080000.0
1,Tech,2025-09-25,AAPL,253.21,257.17,251.71,256.87,55200000.0
2,Tech,2025-09-24,AAPL,255.22,255.74,251.04,252.31,42300000.0
3,Tech,2025-09-23,AAPL,255.88,257.34,253.58,254.43,60280000.0
4,Tech,2025-09-22,AAPL,248.30,256.64,248.12,256.08,105520000.0


In [7]:
stocks.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53011 entries, 0 to 53010
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Industry  53011 non-null  object 
 1   Date      53011 non-null  object 
 2   Ticker    53011 non-null  object 
 3   Open      53011 non-null  float64
 4   High      53011 non-null  float64
 5   Low       53011 non-null  float64
 6   Close     53011 non-null  float64
 7   Volume    53010 non-null  float64
dtypes: float64(5), object(3)
memory usage: 3.2+ MB


## Question 1: Analytical Question
Write a question about your data that can be explored with an interactive plot. A good example would be a question where you can compare different categories. Explain how the plot help answer your question and why you choose that plot type.

Examples:
- Which regions have the highest average values?
- How does one variable compare across time?

**Question**: Overtime, how does the high price of a stock differ from the average high price within the industry?

An interactive plot will help to answer this question by allowing use to isolate specific companies and compare their performance directly against other stocks in the same category. By toggling between industries and individual stocks, we can identify which outperform or underperform compared to their sector's average and see how these trends shift on a monthly basis.

## Question 2. Create a simple interaction plot
Create a plot, it can be related to your question #1 or a new question, that users can interact with in some meaningful way. Explain in a markdown, how does the interaction add to the analysis.

Example of possible interactions:
*   Hover over information
*   Toogle between groups

In [8]:
stocks['Date'] = pd.to_datetime(stocks['Date'])
stocks['Year'] = stocks['Date'].dt.year
stocks['Month'] = stocks['Date'].dt.month

# Select overlapping years
filtered_stocks = stocks[(stocks['Year'] >= 2016) & (stocks['Year'] <= 2024)]

# Calculate monthly highs for each stock/ticker
ticker_highs = filtered_stocks.groupby(['Year', 'Month', 'Industry', 'Ticker'])['High'].max().reset_index()

# Calculate industry average monthly highs
industry_highs = filtered_stocks.groupby(['Year', 'Month', 'Industry'])['High'].mean().reset_index()
industry_highs = industry_highs.rename(columns={'High': 'Avg_High'})

# Merge with ticker highs
combined_df = pd.merge(
    ticker_highs,
    industry_highs,
    on=['Year', 'Month', 'Industry'],
    how='left'
)

# Radio buttons for each industry
industry_radio = pn.widgets.RadioButtonGroup(
    name='Select Industry',
    options=sorted(list(ticker_highs['Industry'].unique())),
    button_type='success'
)

year_slider = pn.widgets.IntSlider(
    name='Select Year',
    start=2016, end=2024, value=2024
)

interactive_highs = combined_df.interactive()

ticker_plot = interactive_highs[
    (interactive_highs['Industry'] == industry_radio) &
    (interactive_highs['Year'] == year_slider)
].sort_values('Month').hvplot(
    x='Month',
    y='High',
    by='Ticker',
    line_width=3,
    marker='o',
    ylabel='High Price (USD)',
    title="Monthly High Price for Individual Stocks vs. Industry Average by Industry",
    width=800,
    height=400,
    grid=True
)

# Industry average line
avg_plot = interactive_highs[
    (interactive_highs['Industry'] == industry_radio) &
    (interactive_highs['Year'] == year_slider)
].sort_values('Month').hvplot(
    x='Month',
    y='Avg_High',
    line_width=3,
    line_dash='dashed',
    color='black',
    label='Industry Avg',
    ylabel='High Price (USD)',
    width=800,
    height=400,
)

# Overlay both plots
stocks_pipeline_plot = (ticker_plot * avg_plot)
stocks_pipeline_plot

**The interactive plot adds to the analysis** of the stock data by allowing us to toggle between and see each industry individually to make observations. By adding an overlap with the average high price during the month across the three chosen stocks, we are also able to visualize how the high price of each stock differs from the average. Hovering over each data point also reveals the stock prices and lets us gauge the difference from the average.

## Question 3. Choropleth Planning
Design a choropleth idea using your dataset.
In a markdown cell:
*  Identify the geographic unit you would map (state, county, country, ZIP code, etc.)
*  Identify the variable you would color by
*  Explain if any aggregation or preprocessing is needed
*  Briefly describe what GeoJSON file would be required

You do not need to have a perfect dataset for this question. However, your plan should be realistic.

If your data does not fully support a choropleth, build a prototype table that explains that structure you would need before mapping.

A realistic choropleth for this project would use **country** as the geographic unit.

The variable we would color by could be:
- **average annual stock return**, or
- **average monthly high price**
for companies grouped by the country where each company is headquartered.

Because our current dataset only contains `Industry`, `Date`, `Ticker`, `Open`, `High`, `Low`, `Close`, and `Volume`, we would need to add a **lookup table** that maps each ticker to a country (or state, if we wanted a U.S.-only map). After that, we could aggregate the stock data by geographic unit and compute a summary statistic such as:
- mean closing price,
- yearly percent return, or
- average trading volume.

For example, the preprocessing steps would be:
1. create a table matching each ticker to a headquarters country,
2. merge that table with the stock dataset,
3. group by country,
4. calculate the metric to visualize.

The GeoJSON file required would be a **world countries GeoJSON** if we map countries, or a **U.S. states GeoJSON** if we instead map company headquarters by state.

## Question 4. Geospatial Possibility Check

Determine whether your dataset can support a map-based visualization.

In a markdown cell, answer one of the following:
- If **yes**, explain what geographic fields you have and what type of map is appropriate.
- If **no**, explain what is missing and what you would need to create a map.

Write code that inspects or prepares the geographic column(s) you may use.

### Geospatial Possibility Check

At the moment, our dataset does **not directly support** a map-based visualization.

The dataset includes the columns:
- `Industry`
- `Date`
- `Ticker`
- `Open`
- `High`
- `Low`
- `Close`
- `Volume`

These fields are useful for time-series and industry comparisons, but they do not include geographic information such as:
- country,
- state,
- city,
- address,
- latitude/longitude, or
- ZIP code.

Because of that, a geospatial visualization is not currently possible using only the stock table. To create a meaningful map, we would need an additional dataset or lookup table that connects each ticker to a geographic location, such as the company headquarters city/state/country. Once that is added, we could build a choropleth or a symbol map showing stock performance by location.

In [9]:
# Inspect columns to check for geographic information
print("Columns in dataset:")
print(stocks.columns.tolist())

# Look for possible geographic columns
geo_keywords = ['state', 'country', 'city', 'zip', 'zipcode', 'address', 'lat', 'latitude', 'lon', 'longitude', 'region']

possible_geo_cols = [col for col in stocks.columns if col.lower() in geo_keywords or any(word in col.lower() for word in geo_keywords)]

print("\nPossible geographic columns found:")
print(possible_geo_cols if possible_geo_cols else "None")

# Show a preview of the dataset structure
stocks.head()

Columns in dataset:
['Industry', 'Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume', 'Year', 'Month']

Possible geographic columns found:
None


,Industry,Date,Ticker,Open,High,Low,Close,Volume,Year,Month
0,Tech,2025-09-26,AAPL,254.10,257.60,253.78,255.46,46080000.0,2025,9
1,Tech,2025-09-25,AAPL,253.21,257.17,251.71,256.87,55200000.0,2025,9
2,Tech,2025-09-24,AAPL,255.22,255.74,251.04,252.31,42300000.0,2025,9
3,Tech,2025-09-23,AAPL,255.88,257.34,253.58,254.43,60280000.0,2025,9
4,Tech,2025-09-22,AAPL,248.30,256.64,248.12,256.08,105520000.0,2025,9


## Question 5. Geopy / Location Preparation

If your dataset has location names, addresses, cities, states, or countries, use geopy on a sample of your data to geocode locations or validate location information.

If your dataset does not have data that contains location, pick 5 destination you want to visit and use geopy to geocode locations.

In [10]:
# Your code . . .

# Import map libraries
import folium
import geopy

In [11]:
# Define a dictionary containing  data
data = {'City':['Boston', 'Philadelphia', 'Columbus', 'Indianapolis', 'Charlotte']}

# Convert the dictionary into DataFrame
df = pd.DataFrame(data)

# Observe the result
df

,City
0,Boston
1,Philadelphia
2,Columbus
3,Indianapolis
4,Charlotte


In [12]:
from geopy.exc import GeocoderTimedOut
from geopy.geocoders import Nominatim

## Function to find the coordinate of a given city
def findGeocode(city):
    ''' try-except is used to overcome the exception thrown by geolocator
    using geocodertimedout '''
    try:
        # Specify the user_agent as your app name and it should not be none
        geolocator = Nominatim(user_agent="CS133")
        return geolocator.geocode(city)
    except GeocoderTimedOut:
        return None

In [13]:
## Declare an empty list to store latitude and longitude values of cities
longitude = []
latitude = []

## Fetch cities and pass through function
for i in (df["City"]):
    if findGeocode(i) != None:
        loc = findGeocode(i)
        ## coordinates returned from function stored into lists
        latitude.append(loc.latitude)
        longitude.append(loc.longitude)

    ## if coordinate for a city not found, insert "NaN" indicating missing value
    else:
        latitude.append(np.nan)
        longitude.append(np.nan)

## Now add the Longitude and Latitude columns to original dataframe
df["Longitude"] = longitude
df["Latitude"] = latitude

df

,City,Longitude,Latitude
0,Boston,-71.057830,42.358834
1,Philadelphia,-75.163526,39.952724
2,Columbus,-83.000707,39.962260
3,Indianapolis,-86.158350,39.768333
4,Charlotte,-80.843083,35.227209


In [ ]:
bostonla = df[(df.City=="Boston")].Latitude.iloc[0] # adding iloc because of the future warning
bostonlo = df[(df.City=="Boston")].Longitude.iloc[0] # adding iloc because of the future warning
print(float(bostonla), float(bostonlo))

42.3588336 -71.0578303


In [ ]:
import folium #import the mapmaker

In [ ]:
boston = folium.Map(location = [float(bostonla), float(bostonlo)],
                zoom_start = 9)
folium.Marker([bostonla, bostonlo],
              popup = 'Boston').add_to(boston)
folium.Marker([df[(df.City=="Philadelphia")].Latitude.iloc[0], df[(df.City=="Philadelphia")].Longitude.iloc[0]],
              popup = 'Philadelphia').add_to(boston)
folium.Marker([df[(df.City=="Columbus")].Latitude.iloc[0], df[(df.City=="Columbus")].Longitude.iloc[0]],
              popup = 'Columbus').add_to(boston)
folium.Marker([df[(df.City=="Indianapolis")].Latitude.iloc[0], df[(df.City=="Indianapolis")].Longitude.iloc[0]],
              popup = 'Indianapolis').add_to(boston)
folium.Marker([df[(df.City=="Charlotte")].Latitude.iloc[0], df[(df.City=="Charlotte")].Longitude.iloc[0]],
              popup = 'Charlotte').add_to(boston)
boston

Our dataset does not include locations so we decided to do the second option of picking five destinations we wanted to visit. We picked five popular cities in the United States that we have not yet visited but want to visit in the future and then used geopy to find the coordinates of them. After that, we used folium to plot those coordinates on a map, choosing to center it around Boston and placing markers on all five major cities.

## Question 6. Panel Widget

Create a Panel Widget that controls something in your analysis such as the ability to choose a column, category, year, etc.

The widget should affect an output such as a plot, table, or summary statistic.

In [ ]:
# Your code . . .
stocks['Difference'] = stocks['Close'] - stocks['Open']
istocks = stocks.interactive()

In [ ]:
stocks.head()

,Industry,Date,Ticker,Open,High,Low,Close,Volume,Year,Month,Difference
0,Tech,2025-09-26,AAPL,254.10,257.60,253.78,255.46,46080000.0,2025,9,1.36
1,Tech,2025-09-25,AAPL,253.21,257.17,251.71,256.87,55200000.0,2025,9,3.66
2,Tech,2025-09-24,AAPL,255.22,255.74,251.04,252.31,42300000.0,2025,9,-2.91
3,Tech,2025-09-23,AAPL,255.88,257.34,253.58,254.43,60280000.0,2025,9,-1.45
4,Tech,2025-09-22,AAPL,248.30,256.64,248.12,256.08,105520000.0,2025,9,7.78


In [ ]:
# Radio buttons for stock price measures
yaxis_stock = pn.widgets.RadioButtonGroup(
    name='Select Price Metric',
    options=['Open', 'High', 'Low', 'Close', 'Difference'],
    button_type='success',
    value='Close' # Default value
)

# Interactive plot using the radio button selection
price_plot = istocks.hvplot.line(
    x='Date',
    y=yaxis_stock,
    by='Ticker',
    line_width=2,
    height=400,
    width=800,
    title='Stock Prices Over Time',
    ylabel='Price (USD)'
)

price_plot

I chose to create an interactive plot that lets viewers switch between stock open, high, low, and close values. All columns represent data that is very similar and thus the four plots look very similar. I also added a fifth column called "Difference" which shows the difference in stock values between Open and Close. We see that as we approach 2015 and later, the open and close values vary very wildly compared to before.

## Question 7. Mini Dashboard

Build a small Panel dashboard with at least two components. Arrange the components so that it is in a readable layout. Your dashboard should include:
* At least one plot,
* An additional element of your choice such as a widget, table, second plot, etc.

In [ ]:
# Write your code here
# Create widget with Tabs()
tabs = pn.Tabs()

tabs.extend([('Stock Price Difference Tracker', price_plot),
             ('Stock Price Changes Across Industry', stocks_pipeline_plot)
            ])

tabs

Tabs
    [0] Interactive(Interactive, name='Stock Price D...)
    [1] Interactive(Interactive, name='Stock Price C...)

## Question 8. Reflection

Write a short reflection addressing all of the following:
- Which interactive element was most useful in your notebook?
- What was the hardest part of working with your dataset?
- Did implementing interactive tools help your analysis? Why or why not?
- If you had more time, what would you improve or add?

### Reflection

The most useful interactive element in our notebook was the ability to switch between stock metrics and industries using widgets. This made it much easier to compare trends without creating a separate graph for every variable or company.

The hardest part of working with our dataset was that it was designed mainly for time-series stock analysis and did not include geographic information. That made some of the mapping tasks more difficult because we had to think about what extra data would be needed instead of directly building a geographic visualization from the stock table.

Implementing interactive tools did help our analysis because the widgets made it easier to focus on one industry or one stock metric at a time. Instead of looking at a crowded static plot, we could explore the data in a more organized way and better notice differences between companies and industry averages.

If we had more time, we would improve the dashboard by adding more filters, cleaner styling, and a geographic component based on company headquarters or market location. We would also consider adding summary statistics and annotations so the dashboard could better explain important stock trends over time.